# 가상문서 임베딩 


In [40]:

import logging 
from typing import List 
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


In [41]:
COLLECTION_NAME = "invest_index"
QDRANT_URL = "http://localhost:6333"

In [42]:
def get_loader(file_path: str):
    try: 
        docs = []   
        loaders = [TextLoader(file_path)]
        for loader in loaders:
            docs.extend(loader.load())
        return docs 
    except FileExistsError as e:
        print(f"failed load file : {file_path}")
        return []


def get_recursive_splitter(chunk_size :int=1000, chunk_overlap: int=200):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def split_docs(docs):
    recursive_spliter = get_recursive_splitter()
    return recursive_spliter.split_documents(docs) 

def get_embedding():
    return OpenAIEmbeddings(
        model="bge-m3",
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
        check_embedding_ctx_length=False)


def initialize_vector_store(docs: List[Document]) -> QdrantVectorStore:
    split_documents = split_docs(docs)

    return QdrantVectorStore.from_documents(
        documents=split_documents,
        embedding=get_embedding(),
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
    )

def create_vector_store() -> QdrantVectorStore:
    return QdrantVectorStore.from_existing_collection(
        embedding=get_embedding(),
        url=QDRANT_URL,
        collection_name=COLLECTION_NAME,
    )


def add_documents_to_vector_store(vector_store: QdrantVectorStore, docs: List[Document]) -> None:
    vector_store.add_documents(docs)



In [ ]:
file_path = "./data/How_to_invest_money.txt" 
docs = get_loader(file_path)
vector_store = initialize_vector_store(docs)


In [45]:
from langchain_core.output_parsers import StrOutputParser 
from langchain_core.prompts import ChatPromptTemplate 
from langchain_openai import OpenAI
from langchain_core.runnables import RunnablePassthrough, RunnableLambda 
from langchain_text_splitters import CharacterTextSplitter 

def call_model(model : str="google/gemma-4-eb", temperature:float=0.5):
    return OpenAI(
        model=model, 
        temperature=temperature,
        base_url="http://localhost:1234/v1",
        api_key="lm-studio"
    )

def create_virtual_doc_chain():
    system = "당신은 고도로 숙련된 AI입니다."
    user = """
    주어진 질문 '{query}'에 대해 직접적으로 답변하는 가상의 문서를 생성하십시오. 
    문서의 크기는 '{chunk_size}'글자 언저리여야 합니다. 
    """
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", user)])
    
    llm = call_model()
    return prompt | llm | StrOutputParser()

    

In [52]:
retriever = vector_store.as_retriever()

def create_retrieval_chain():
    return RunnableLambda(lambda  x : retriever.invoke(x['virtual_doc']))

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def create_final_response_chain():
    final_prompt = ChatPromptTemplate.from_template("""
    다음 정보의 질문을 바탕으로 답변해주세요: 
    컨텍스트 : {context} 
    질문 : {question} 
    답변 : """
    )
    final_llm = call_model()
    return final_prompt | final_llm 

In [49]:
from langchain_core.runnables import RunnableLambda

def print_input_output(input_data, output_data, step_name):
    print(f"\n--- {step_name} ---")
    print(f"Input: {input_data}")
    print(f"Output: {output_data}")
    print("-" * 50)

In [53]:
def create_pipeline_with_logging():
    virtual_doc_chain = create_virtual_doc_chain()
    retrieval_chain = create_retrieval_chain()
    final_response_chain = create_final_response_chain()

    def virtual_doc_step(x):
        result = {"virtual_doc": virtual_doc_chain.invoke({
            "query": x["question"],
            "chunk_size": 200
        })}
        print_input_output(x, result, "Virtual Doc Generation")
        return {**x, **result}

    # 문서 검색 단계
    def retrieval_step(x):
        result = {"retrieved_docs": retrieval_chain.invoke(x)}
        print_input_output(x, result, "Document Retrieval")
        return {**x, **result}

    # 컨텍스트 포맷팅 단계
    def context_formatting_step(x):
        result = {"context": format_docs(x["retrieved_docs"])}
        print_input_output(x, result, "Context Formatting")
        return {**x, **result}

    # 최종 응답 생성 단계
    def final_response_step(x):
        result = final_response_chain.invoke(x)
        print_input_output(x, result, "Final Response Generation")
        return result

    # 전체 파이프라인 구성
    pipeline = (
        RunnableLambda(virtual_doc_step)
        | RunnableLambda(retrieval_step)
        | RunnableLambda(context_formatting_step)
        | RunnableLambda(final_response_step)
    )

    return pipeline

# 파이프라인 객체 생성
pipeline = create_pipeline_with_logging()

In [55]:
# 예시 질문과 답변
question = "주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?"
response = pipeline.invoke({"question": question})
print(f"최종 답변: {response}")


--- Virtual Doc Generation ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?'}
Output: {'virtual_doc': "문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언"}
--------------------------------------------------

--- Document Retrieval ---
Input: {'question': '주식 시장의 변동성이 높을 때 투자 전략은 무엇인가요?', 'virtual_doc': "문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. \n    문서의 크기는 '200'글자 언저리여야 합니다. 